# 02 — Exploração do Pipeline de Pré-processamento

Este notebook tem dois objetivos:
1. **Visualizar** cada etapa do pipeline em imagens individuais
2. **Comparar** variantes de configuração usando SSIM e PSNR

Todas as configurações são lidas do `config.yaml` na raiz do projeto.
Para mudar parâmetros, edite o YAML — não este notebook.

In [ ]:
import sys
sys.path.insert(0, '..')  # permite importar src/ a partir da pasta notebooks/

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import load_image, show_comparison
from src.preprocessing import Pipeline, shades_of_gray, sharp_razor, apply_clahe
from src.evaluation import calculate_ssim_psnr, evaluate_pipeline_variants

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

pipeline = Pipeline(cfg)
print('Pipeline carregado. Config de pré-processamento:')
print(yaml.dump(cfg['preprocessing'], allow_unicode=True))

## 1. Carregar imagem de teste

In [ ]:
# Altere o caminho para qualquer imagem do ISIC 2018
IMAGE_PATH = '../data/raw/ISIC_0000074.jpg'

img_original = load_image(IMAGE_PATH)
print(f'Shape: {img_original.shape} | dtype: {img_original.dtype}')

plt.figure(figsize=(5, 5))
plt.imshow(img_original[:, :, ::-1])  # BGR -> RGB
plt.title('Imagem original')
plt.axis('off')
plt.show()

## 2. Visualizar cada etapa do pipeline

In [ ]:
img_cc   = pipeline.step_color_constancy(img_original)
img_hr   = pipeline.step_hair_removal(img_cc)
img_cl   = pipeline.step_clahe(img_hr)
img_rs   = pipeline.step_resize(img_cl)

show_comparison({
    'Original':         img_original,
    '1. Color Constancy': img_cc,
    '2. Hair Removal':  img_hr,
    '3. CLAHE':         img_cl,
    '4. Resize 256×256': img_rs,
}, figsize=(20, 4))

## 3. Busca pelo melhor clip_limit do CLAHE

Itera sobre uma faixa de valores e calcula SSIM/PSNR em relação à imagem original.

In [ ]:
exp_cfg = cfg['experiments']['clip_limit_search']
clip_limits = np.arange(exp_cfg['start'], exp_cfg['stop'], exp_cfg['step'])

rows = []
for clip in clip_limits:
    processed = apply_clahe(img_hr, clip_limit=float(clip))
    m = calculate_ssim_psnr(img_original, processed)
    rows.append({'clip_limit': round(float(clip), 2), **m})

df_clip = pd.DataFrame(rows)
df_clip.to_csv('../results/metrics/clip_limit_search.csv', index=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(df_clip['clip_limit'], df_clip['ssim'], marker='o', color='steelblue')
ax1.set(xlabel='clip_limit', ylabel='SSIM', title='SSIM vs clip_limit')
ax1.axvline(x=cfg['preprocessing']['clahe']['clip_limit'], color='red',
            linestyle='--', label=f"atual ({cfg['preprocessing']['clahe']['clip_limit']})")
ax1.legend()

ax2.plot(df_clip['clip_limit'], df_clip['psnr'], marker='o', color='darkorange')
ax2.set(xlabel='clip_limit', ylabel='PSNR (dB)', title='PSNR vs clip_limit')
ax2.axvline(x=cfg['preprocessing']['clahe']['clip_limit'], color='red',
            linestyle='--', label=f"atual ({cfg['preprocessing']['clahe']['clip_limit']})")
ax2.legend()

plt.tight_layout()
plt.savefig('../results/figures/clip_limit_search.png', dpi=150)
plt.show()

print('\nResultados:')
display(df_clip)

## 4. Comparação entre variantes da ordem do pipeline

In [ ]:
img_cc = pipeline.step_color_constancy(img_original)
img_hr = pipeline.step_hair_removal(img_cc)
img_cl_only = pipeline.step_clahe(img_cc)

variants = {
    'cc → hr → clahe': pipeline.step_clahe(img_hr),
    'cc → clahe → hr': pipeline.step_hair_removal(img_cl_only),
    'cc → hr (sem clahe)': img_hr,
}

results = evaluate_pipeline_variants(img_original, variants)
df_variants = pd.DataFrame(results)
df_variants.to_csv('../results/metrics/pipeline_variants.csv', index=False)

print('Comparação entre variantes:')
display(df_variants)

show_comparison({'Original': img_original, **variants}, figsize=(18, 4))